# Segmentation Spatiale Éco-Épidémiologique - Ébola RDC
Ce notebook réalise le partitionnement des zones de santé en fonction de critères sanitaires et environnementaux.

In [1]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

# Inclusion du dossier 'src' dans le chemin système
sys.path.append('../')
from src.models import EcoEbolaClustering

In [2]:
# Chargement des données préparées
df_zone = pd.read_csv('../data/donnees_zones_preparees.csv', sep=';')

In [3]:
# Recherche du nombre optimal de clusters (Méthode du coude)
from sklearn.cluster import KMeans
model_explorer = EcoEbolaClustering()
scaled_data = model_explorer.fit_prepare_data(df_zone)

wcss = []
sil_scores = []
range_n_clusters = range(2, 7)

for i in range_n_clusters:
    km = KMeans(n_clusters=i, random_state=42, n_init=10)
    labels = km.fit_predict(scaled_data)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(scaled_data, labels))

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel('Nombre de Clusters')
ax1.set_ylabel('WCSS', color='tab:blue')
ax1.plot(range_n_clusters, wcss, marker='o', color='tab:blue', label='WCSS')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Score de Silhouette', color='tab:red')
ax2.plot(range_n_clusters, sil_scores, marker='s', color='tab:red', label='Silhouette')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Optimisation du Nombre de Clusters')
plt.grid(True, linestyle=':')
plt.show()

NameError: name 'EcoEbolaClustering' is not defined

In [4]:
# Application de K-Means et Clustering Hiérarchique
n_optimal = 3
pipeline = EcoEbolaClustering(n_clusters=n_optimal)

# Exécution des algorithmes
df_results = pipeline.apply_kmeans(df_zone)
df_results = pipeline.apply_hierarchical(df_results)

# Comparaison des scores de silhouette
scaled_data = pipeline.scaler.transform(df_zone[pipeline.features])
score_km = silhouette_score(scaled_data, df_results['cluster_kmeans'])
score_hc = silhouette_score(scaled_data, df_results['cluster_hierarchical'])

print(f"Silhouette Score (K-Means): {score_km:.4f}")
print(f"Silhouette Score (Hierarchical): {score_hc:.4f}")

In [5]:
# Extraction des profils pour K-Means (souvent plus stable)
profils = pipeline.get_cluster_profiles(df_results, 'cluster_kmeans')
print("\n--- Profils Moyens des Clusters (K-Means) ---")
print(profils.round(2))

# Sauvegarde du modèle
pipeline.save_model('../models/ebola_clustering_final.pkl')

In [6]:
# Visualisation Spatiale des Clusters
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=df_results, 
    x='longitude', 
    y='latitude', 
    hue='cluster_kmeans', 
    style='cluster_hierarchical', # Permet de voir les différences entre les deux méthodes
    palette='viridis', 
    size='total_cases', 
    sizes=(50, 500),
    alpha=0.7
)
plt.title('Segmentation Spatiale des Zones de Santé (K-Means vs CHA)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Clusters (Hue=KM, Style=CHA)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()